# Concrete Compressive Strength - Linear Regression

This notebook continues from the existing project preprocessing and feature engineering notebooks. It reads `data/processed/concrete_data_cleaned.csv`; EDA and dataset cleaning are not repeated.

The training pipeline fits feature selection, scaling and PCA on training folds and saves those fitted transformations with the model so the frontend can predict from the original eight inputs.

## 1. Import Libraries

In [ ]:
from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Folder Location

In [ ]:
PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "data/processed/concrete_data_cleaned.csv").is_file()
)
MODEL_FOLDER = PROJECT_ROOT / "models/trained"
REPORT_FOLDER = PROJECT_ROOT / "reports/evaluation/linear_regression"
MODEL_FOLDER.mkdir(parents=True, exist_ok=True)
REPORT_FOLDER.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)


## 3. Load Cleaned Dataset

In [ ]:
df = pd.read_csv(PROJECT_ROOT / "data/processed/concrete_data_cleaned.csv")
display(df.head())
print("Dataset shape:", df.shape)

## 4. Feature and Target Separation

In [ ]:
target_column = "compressive_strength"
X = df.drop(columns=target_column)
y = df[target_column]
display(X.head())
display(y.head())
print("X shape:", X.shape)
print("y shape:", y.shape)

## 5. Train-Test Split
80% training, 20% testing; random_state=42.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## 6. Linear Regression Pipeline

In [ ]:
linear_pipeline = Pipeline([
    ("selection", VarianceThreshold(threshold=0.01)),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95)),
    ("model", LinearRegression())
])
print(linear_pipeline)

## 7. Evaluation Function

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    test_predictions = model.predict(X_test)
    train_predictions = model.predict(X_train)
    return {
        "MAE": mean_absolute_error(y_test, test_predictions),
        "MSE": mean_squared_error(y_test, test_predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, test_predictions)),
        "R²": r2_score(y_test, test_predictions),
        "Train_R²": r2_score(y_train, train_predictions)
    }, test_predictions

## 8. Linear Regression - Baseline Training and Evaluation

In [ ]:
linear_results, linear_predictions = evaluate_model(
    linear_pipeline, X_train, X_test, y_train, y_test
)
print(pd.Series(linear_results).round(4))

linear_model = linear_pipeline.named_steps["model"]
coefficients = pd.DataFrame({
    "Principal Component": [f"PC{i+1}" for i in range(linear_pipeline.named_steps["pca"].n_components_)],
    "Coefficient (scaled inputs)": linear_model.coef_
})
display(coefficients)
print("Intercept:", round(linear_model.intercept_, 4))

In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=linear_predictions, alpha=0.75)
minimum = min(y_test.min(), linear_predictions.min())
maximum = max(y_test.max(), linear_predictions.max())
plt.plot([minimum, maximum], [minimum, maximum], "r--", label="Perfect prediction")
plt.xlabel("Actual Strength (MPa)")
plt.ylabel("Predicted Strength (MPa)")
plt.title("Linear Regression: Actual vs Predicted")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Five-Fold Cross-Validation

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
negative_mse_scores = cross_val_score(
    linear_pipeline, X_train, y_train,
    cv=cv, scoring="neg_mean_squared_error"
)
fold_rmse = np.sqrt(-negative_mse_scores)
print("Each fold RMSE:", fold_rmse)
print("Mean CV RMSE:", fold_rmse.mean())
print("CV RMSE Standard Deviation:", fold_rmse.std())
pd.DataFrame({"Fold": range(1, 6), "RMSE": fold_rmse}).to_csv(
    REPORT_FOLDER / "baseline_cv_metrics.csv", index=False
)

## 10. Hyperparameter Tuning

In [ ]:
linear_parameter_grid = {"model__fit_intercept": [True, False]}
linear_grid_search = GridSearchCV(
    linear_pipeline,
    linear_parameter_grid,
    cv=cv,
    scoring="neg_mean_squared_error"
)
linear_grid_search.fit(X_train, y_train)
print("Best Linear Regression parameters:", linear_grid_search.best_params_)
print("Best Linear Regression CV RMSE:", np.sqrt(-linear_grid_search.best_score_))
best_model = linear_grid_search.best_estimator_

## 11. Final Evaluation

In [ ]:
final_results, final_predictions = evaluate_model(
    best_model, X_train, X_test, y_train, y_test
)
print(pd.Series(final_results).round(4))
pd.DataFrame([final_results]).to_csv(REPORT_FOLDER / "test_metrics.csv", index=False)

## 12. Actual vs Predicted and Residuals
Residual = Actual − Predicted.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(y_test, final_predictions, alpha=0.7)
minimum = min(y_test.min(), final_predictions.min())
maximum = max(y_test.max(), final_predictions.max())
axes[0].plot([minimum, maximum], [minimum, maximum], "r--")
axes[0].set_xlabel("Actual Strength (MPa)")
axes[0].set_ylabel("Predicted Strength (MPa)")
axes[0].set_title("Linear Regression: Actual vs Predicted")
axes[1].scatter(final_predictions, y_test.to_numpy() - final_predictions, alpha=0.7)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_xlabel("Predicted Strength (MPa)")
axes[1].set_ylabel("Residual (MPa)")
plt.tight_layout()
plt.savefig(REPORT_FOLDER / "evaluation_plot.png", dpi=150)
plt.show()

## 13. Final Prediction

In [ ]:
# Use one real test observation while preserving the training column order.
sample_input = X_test.iloc[[0]].copy()
sample_prediction = best_model.predict(sample_input)[0]

display(sample_input)
print(f"Predicted Concrete Compressive Strength: {sample_prediction:.2f} MPa")
print(f"Actual strength for this test observation: {y_test.iloc[0]:.2f} MPa")

## 14. Save Linear Regression Model

In [ ]:
# Save the complete fitted pipeline: feature selection, scaling, PCA and model.
MODEL_PATH = MODEL_FOLDER / "linear_regression_pca_pipeline.joblib"
joblib.dump(best_model, MODEL_PATH, compress=3)
print("Model saved:", MODEL_PATH)

## 15. Save Results and Model Details

In [ ]:
prediction_table = pd.DataFrame({
    "Actual Strength": y_test.to_numpy(),
    "Predicted Strength": final_predictions
})
prediction_table.to_csv(REPORT_FOLDER / "test_predictions.csv", index=False)

model_details = {
    "model": "Linear Regression",
    "feature_columns": X.columns.tolist(),
    "target": target_column,
    "train_rows": len(X_train),
    "test_rows": len(X_test),
    "best_parameters": linear_grid_search.best_params_,
    "test_metrics": final_results,
    "versions": {"scikit-learn": sklearn.__version__, "numpy": np.__version__,
                 "pandas": pd.__version__, "joblib": joblib.__version__}
}
metadata_folder = PROJECT_ROOT / "models/metadata"
metadata_folder.mkdir(parents=True, exist_ok=True)
with open(metadata_folder / "linear_regression_metadata.json", "w") as file:
    json.dump(model_details, file, indent=4)
print("Results saved.")